一、依赖安装

! 表示：把后面的内容当作系统命令执行;<br>
- tokenizers是 Hugging Face 提供的分词器库。
- tqdm是进度条库，用来显示“程序已经处理了多少数据”


二、分块读取工具函数[获取位置]

伪代码
```python
函数：find_chunk_boundaries

输入：
    一个文件
    希望分成多少块
    分隔符

先获取文件总大小
把文件指针移回开头

对每一个中间边界：
    先猜一个位置
    从这个位置开始往后找
    每次读取x个字节

    如果找到特殊 token：
        把边界设在它的位置
        停止

    如果读到文件末尾：
        把边界设在文件末尾
        停止

    如果没找到：
        把扫描位置向后移动 4096
        继续找
```

In [1]:
import os
from typing import BinaryIO

def find_chunk_boundaries(
    file: BinaryIO,                 #二进制
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:                     #返回一个整数列表（存储切块的边界位置），边界数组的长度 = 块数 + 1
    """
    将文件切分为可以独立计数的块。
    如果边界发生重叠（例如文件太小或分隔符太稀疏），返回的块数量可能会少于预期。
    """
    # 确保传入的分隔符是字节串类型，因为文件是以二进制模式读取的
    assert isinstance(split_special_token, bytes), "必须使用字节串（bytes）表示特殊 Token"

    # 获取文件的总字节大小
    file.seek(0, os.SEEK_END) #seek：把文件阅读位置移动到某个地方；os.SEEK_END：从文件末尾开始定位
    file_size = file.tell()     #告诉我当前文件位置是多少
    file.seek(0) # 回到文件开头

    # 计算初步的理想块大小（字节数）
    chunk_size = file_size // desired_num_chunks

    # 初始的边界猜测：根据块大小进行均匀分布
    # 边界数组包含起始位置 0 和结束位置 file_size
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size # 确保最后一个边界精确指向文件末尾

    mini_chunk_size = 4096  # 每次向后搜索的缓冲区大小4k字节

    # 遍历除了开头和结尾之外的所有中间边界点
    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # 跳转到初步猜测的边界位置
        
        while True:
            # 读取一小块数据进行扫描
            mini_chunk = file.read(mini_chunk_size)      #seek() 决定“从哪里开始读”，read() 决定“读多少”

            # 如果读到了文件末尾（EOF），说明后面没有分隔符了，直接设为文件末尾
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # 在当前小块数据中查找指定的分隔符
            found_at = mini_chunk.find(split_special_token)
            
            if found_at != -1:
                # 如果找到了分隔符，将边界调整到该分隔符的确切位置
                chunk_boundaries[bi] = initial_position + found_at
                break
            
            # 如果没找到，继续向后移动指针进行下一轮搜索
            initial_position += mini_chunk_size

    # set()：去除重复的边界，防止多个猜测点指向同一个分隔符
    # sorted()：确保边界按从小到大的顺序排列
    return sorted(set(chunk_boundaries))

三、按块读取文本

In [ ]:
import time
from tqdm import tqdm

def iter_text_chunks_with_monitor(
    file_path: str,
    chunk_size: int = 1_000_000,  # 1MB
    log_every: int = 5,          # 每N个chunk打印一次
):
    start_time = time.time() #记录函数开始运行的时间
    bytes_processed = 0
    chunk_count = 0

    with open(file_path, "r", encoding="utf-8") as f: #`with`语法：代码块结束自动关闭文件，不用手动写 close ()，安全
        buffer = []                 # 暂时保存多行文本
        buffer_size = 0

        for line in f:               #一行一行读取
            buffer.append(line)
            buffer_size += len(line)
            bytes_processed += len(line)

            # 当缓冲区大小达到或超过指定的 chunk_size 时，输出当前缓冲区内容作为一个 chunk
            if buffer_size >= chunk_size:
                yield "".join(buffer) # `yield`把拼接好的整块文本返回给调用者,不会结束函数！函数暂停在这里；外面 for 循环下一次迭代的时候，函数从这一行往下继续跑
                buffer = []
                buffer_size = 0
                chunk_count += 1

                if chunk_count % log_every == 0:
                    log_status(
                        prefix="分词器流式处理",
                        bytes_processed=bytes_processed,
                        start_time=start_time,
                    )

        # 剩下残余文本拼接 yield 输出。保证文件末尾内容不会被丢掉
        if buffer:
            yield "".join(buffer)


1. **大文件不要 read ()**：`f.read()`一次性加载全部，大文件内存爆炸；`for line in f`逐行读，内存友好。
2. `yield`生成器：函数返回一部分数据，暂停，外部循环驱动继续执行，不用把全部结果存列表。
3. buffer 缓冲区模式：一行行攒，到达阈值输出一块；**文件结束一定要处理 buffer 残余，防止丢数据**。
4. `bytes_processed`统计的是字符数不是真实磁盘字节；中文场景只是近似值。
5. log_every 取模打印日志，避免每一块都打印造成控制台刷屏。
6. `with open()`自动关闭文件，写文件读取优先用 with。

四、训练BPE分词器

Step1:在不修改tokenizer内部实现的前提下，实时监控内存占用与数据吞吐量，理解tokenizer训练的真实系统行为。

In [4]:
# 安装监控需要的依赖
!pip install psutil

In [5]:
# 当前进程所占内存
import psutil
import os

def get_memory_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

In [6]:
# 日志状态函数输出（内存占用、处理数据量、处理速度）
import time
def log_status(prefix, bytes_processed, start_time):
    elapsed = time.time() - start_time  
    mb = bytes_processed / 1024 / 1024
    throughput = mb / elapsed if elapsed > 0 else 0.0 # 计算吞吐量即每秒处理的数据量
    mem = get_memory_mb() 

    print(
        f"{prefix} | "
        f"mem={mem:7.1f} MB | "
        f"data={mb:8.1f} MB | "
        f"speed={throughput:6.2f} MB/s"
    )

Step2:训练BPE Tokenizer

In [7]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder #解码器，把字节还原回原始文本
from tokenizers.normalizers import NFKC  # 标准化，把全角字符、等价 unicode 字符归一化，中文、符号清洗用
#from tqdm import tqdm

1. Tokenizer(核心容器)
- normalizer  = NFKC()          # 第1步：字符归一化
- pre_tokenizer = ByteLevel()   # 第2步：切初始单元（GPT2用字节）
- model = BPE(...)              # 第3步：BPE合并算法
- decoder = ByteLevelDecoder()  # 第4步：解码还原
2. `train_from_iterator(iterator, trainer)`：**从文本迭代器训练分词器**
- 内部循环：不断从`text_iterator`拿文本块；
- 统计：文本切分成基础单元（ByteLevel 字节），统计相邻子词对的出现频率；
- BPE 贪心合并：反复合并最高频子词对，直到词表大小达到`trainer`指定的`vocab_size`；
- 训练结束后，**直接修改 `tokenizer` 对象内部状态**：词表、合并规则全部写入这个 tokenizer 对象。
> 伪理解：`tokenizer`这个对象，训练前是空壳，调用`.train_from_iterator()`之后，壳里面装入学习好的 BPE 词表与合并规则。

In [ ]:
def train_bpe_tokenizer(
    train_file: str,
    val_file: str | None = None, #验证集文本，注意：BpeTrainer 本身不会用验证集做训练，这里只是把验证集文本一起喂进训练迭代器！不是做验证评估，只是扩充训练数据
    vocab_size: int,
    output_dir: str # 输出保存目录
):
    os.makedirs(output_dir, exist_ok=True) #创建输出文件夹；`exist_ok=True` 如果文件夹已存在不抛异常。

    special_tokens = [ #这些 token**不会被 BPE 切分**，训练时直接固定加入词表，有独立 id
        "<|endoftext|>", # GPT‑2 原生文档结束符
        "<|unk|>", # 未知字符
        "<|pad|>", # padding填充
        "<|bos|>", # 开头
        "<|eos|>", # 结尾
    ]

    tokenizer = Tokenizer(BPE(unk_token="<|unk|>"))
    # 预处理，规范化器：把全角字符、等价 unicode 字符归一化，中文、符号清洗用
    tokenizer.normalizer = NFKC()
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)# 预分词器：先把文本转成 UTF-8 字节，所有字符映射到 0-255 的基础字节，`add_prefix_space=True`：**单词前面自动加空格**
    tokenizer.decoder = ByteLevelDecoder()# 解码器：把字节还原回原始文本，和预分词配对
    # BPE 训练器：负责执行 BPE 合并算法
    trainer = BpeTrainer( 
        vocab_size=vocab_size,
        special_tokens=special_tokens,
        show_progress=True, 
    )

    def text_iterator():
        # 训练集
        for chunk in iter_text_chunks_with_monitor(
            train_file,
            chunk_size=1_000_000,    # 每次处理大小约为1MB的原始数据，处理完就会清除数据方便继续处理
            log_every=20,
        ):
            yield chunk

        # 验证集（可选）
        if val_file is not None:
            for chunk in iter_text_chunks_with_monitor(
                val_file,
                chunk_size=1_000_000,
                log_every=10,
            ):
                yield chunk

    print("🚀 开始训练BPE Tokenizer...")
    #tokenizer：前面构造好的 `tokenizers.Tokenizer` **实例对象**
    # `train_from_iterator(iterator, trainer)`：**从文本迭代器训练分词器**`：**对象的成员方法（method）**，属于`tokenizers`库 AP
    tokenizer.train_from_iterator(text_iterator(), trainer=trainer)
    print("✅ BPE Tokenizer训练完成")

    tokenizer.save(os.path.join(output_dir, "tokenizer.json"))
    print(f"💾 分词器已保存至{output_dir}/tokenizer.json")

    return tokenizer

# 主函数
if __name__ == "__main__":
    train_path = "./data/TinyStoriesV2-GPT4-train.txt"
    val_path = "./data/TinyStoriesV2-GPT4-valid.txt"
    tokenizer = train_bpe_tokenizer( #把返回的 Tokenizer 实例存到变量`tokenizer`里
        train_file=train_path,
        val_file=val_path,
        vocab_size=50257,     # 词表大小（通常设为32000或50257等）
        output_dir="./bpe_tokenizer",
    )

🚀 开始训练BPE Tokenizer...
分词器流式处理 | mem=  117.3 MB | data=    19.1 MB | speed= 33.44 MB/s
分词器流式处理 | mem=  140.3 MB | data=    38.2 MB | speed= 39.21 MB/s
分词器流式处理 | mem=  165.9 MB | data=    57.2 MB | speed= 40.97 MB/s
分词器流式处理 | mem=  188.8 MB | data=    76.3 MB | speed= 41.77 MB/s
分词器流式处理 | mem=  211.7 MB | data=    95.4 MB | speed= 42.43 MB/s
分词器流式处理 | mem=  236.6 MB | data=   114.5 MB | speed= 42.69 MB/s
分词器流式处理 | mem=  259.5 MB | data=   133.5 MB | speed= 43.65 MB/s
分词器流式处理 | mem=  282.4 MB | data=   152.6 MB | speed= 44.15 MB/s
分词器流式处理 | mem=  305.2 MB | data=   171.7 MB | speed= 43.52 MB/s
分词器流式处理 | mem=  328.1 MB | data=   190.8 MB | speed= 44.02 MB/s
分词器流式处理 | mem=  353.0 MB | data=   209.8 MB | speed= 43.05 MB/s
分词器流式处理 | mem=  374.0 MB | data=   228.9 MB | speed= 43.35 MB/s
分词器流式处理 | mem= 2261.6 MB | data=   248.0 MB | speed=  6.57 MB/s
分词器流式处理 | mem= 2077.3 MB | data=   267.1 MB | speed=  6.90 MB/s
分词器流式处理 | mem= 2077.3 MB | data=   286.1 MB | speed=  6.97 MB/s
分词器流式处理 | mem= 20

tokenizer.json(token-id字典)：
1. 先把文本拆成最小单元（字符/byte）
2. 看 merges 里哪些 pair 最优先合并
3. 反复合并到不能再合并为止
4. 最后得到的 token 去 vocab 里查 id

五、验证训练的BPE Tokenizer

In [11]:
encoded = tokenizer.encode(" Hello, world! <|endoftext|>")
print(encoded.tokens)    # 打印编码后的token序列
print(encoded.ids)       # 打印编码后的ID序列
# print(tokenizer.decode([1501])) # 输出应该是" world"（前面带个空格）

# 将Ġ替换回空格
clean_tokens = [t.replace('Ġ', ' ') for t in encoded.tokens]
print(clean_tokens)

['ĠHello', ',', 'Ġworld', '!', 'Ġ', '<|endoftext|>']
[8431, 16, 1501, 5, 149, 0]
[' Hello', ',', ' world', '!', ' ', '<|endoftext|>']


Q: 为什么在GTP-2正则化风格中训练出来的tokenizer会在单词前添加一个'Ġ'？
A: 因为对于模型而言，在同一个单词中，没有带空格和带有空格表示的Token是不同的，因此为了区分这两种情况就会添加了一个'Ġ'来表示带空格的Token（比如"hello"和"Ġhello"，前者表示"hello"这个单词，后者表示" hello"这个单词）。

In [1]:
# Token统计函数
def analyze_tokenizer(tokenizer, texts):
    lengths = [len(tokenizer.encode(t).ids) for t in texts]
    return {
        "avg_tokens": sum(lengths) / len(lengths), # 平均处理token数
        "max_tokens": max(lengths),                # 最大token数，用于设置最大处理序列长度（决定是否截断序列处理）
    }

In [ ]:
import random

def load_stories(file_path, num_samples=None):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    stories = [s.strip() for s in content.split("<|endoftext|>") if s.strip()]
    
    if num_samples is not None:
        n = min(num_samples, len(stories))
        stories = random.sample(stories, k=n)
    return stories

train_path = "./data/TinyStoriesV2-GPT4-train.txt"
val_path = "./data/TinyStoriesV2-GPT4-valid.txt"
test_texts = load_stories(val_path, num_samples=10)
train_texts = load_stories(train_path, num_samples=20)
print(f"随机抽取的训练样本数: {len(train_texts)}")
print(f"随机抽取的验证样本数: {len(test_texts)}")

# 分析训练集和验证集的token统计
train_stats = analyze_tokenizer(tokenizer, train_texts)
val_stats = analyze_tokenizer(tokenizer, test_texts)
print("训练集统计:", train_stats)
print("验证集统计:", val_stats)